In [1]:
# Exemplo de uso da biblioteca pns-womens-health-analysis
# 
# Objetivo deste notebook:
# - Mostrar como listar as variáveis semânticas disponíveis no mapping.py
# - Demonstrar o uso da função de alto nível `get_dataframe`
# - Verificar o comportamento de cache (primeira vs segunda execução)
#
# Observação importante:
# - Antes de rodar, garanta que:
#   - O arquivo .env contém BILLING_PROJECT_ID válido
#   - Você está autenticado no Google Cloud (gcloud auth application-default login)

In [2]:

import pandas as pd
import sys
from pathlib import Path

# adiciona a raiz do projeto ao PYTHONPATH
sys.path.append(str(Path("..").resolve()))

from service.pns_service import get_dataframe, list_variables

In [3]:
# 1. Explorar variáveis disponíveis no mapping.py
print("Variáveis semânticas disponíveis (todas as origens):")
df_vars = list_variables()
print(df_vars)


Variáveis semânticas disponíveis (todas as origens):
                   variable code_2013 type_2013  exists_2013 code_2019  \
0               anos_estudo   vdd004a       int         True   vdd004a   
1              estado_civil      c011    string         True      c011   
2              filhos_vivos      r045       int         True      r045   
3                     idade      c008       int         True      c008   
4     identificador_unidade   upa_pns    string         True   upa_pns   
5             ja_engravidou      r039    string         True      r039   
6                mamografia      r015    string         True      r015   
7          mamografia_pagou      r019    string         True      r019   
8   medico_pediu_mamografia      r014    string         True      r014   
9             peso_amostral    v00291     float         True    v00291   
10               preventivo      r001    string         True    r00101   
11                     raca      c009    string         Tru

In [4]:
# 2. Definir variáveis e origens para análise
variables = [
    "sexo", "idade", "mamografia", "preventivo",
    "renda_per_capita", "peso_amostral",
]

sources = ["2013", "2019"]

In [7]:
# 3. Primeiro acesso: dispara download + transformação + cache
print("\n=== PRIMEIRA EXECUÇÃO (deve baixar do BigQuery e popular o cache) ===")
df = get_dataframe(variables=variables, sources=sources)
print("Shape do DataFrame:", df.shape)
print("Colunas:", list(df.columns))
print("Amostra:")
print(df.head())


=== PRIMEIRA EXECUÇÃO (deve baixar do BigQuery e popular o cache) ===
Shape do DataFrame: (14007, 8)
Colunas: ['renda_per_capita', 'mamografia', 'identificador_unidade', 'origem', 'peso_amostral', 'sexo', 'idade', 'preventivo']
Amostra:
   renda_per_capita     mamografia identificador_unidade origem  \
0            1078.0            sim               1100001   2013   
1            1233.0  não informado               1100002   2013   
2             678.0  não informado               1100003   2013   
3            2000.0            não               1100004   2013   
4             534.0  não informado               1100005   2013   

   peso_amostral sexo  idade preventivo  
0     449.598170    2     64          4  
1     351.427625    2     36          3  
2     662.672345    2     62          5  
3     129.832870    2     49          1  
4     834.165592    2     52          4  


In [6]:
# 4. Segundo acesso: deve ser muito mais rápido (apenas leitura do SQLite)
print("\n=== SEGUNDA EXECUÇÃO (deve ler apenas do cache local) ===")
df_cache = get_dataframe(variables=variables, sources=sources)
print("Shape do DataFrame (cache):", df_cache.shape)
print("Mesmas colunas?:", list(df_cache.columns) == list(df.columns))



=== SEGUNDA EXECUÇÃO (deve ler apenas do cache local) ===
Shape do DataFrame (cache): (14007, 8)
Mesmas colunas?: True


In [ ]:
import numpy as np

variables = ["mamografia_fez", "preventivo_fez"]
df2 = get_dataframe(variables=variables, sources=sources)

df2["preventivo_flag"] = np.where(
    df2["preventivo"] == "5",
    "Não fez",
    "Fez"
)

df2["mamografia_flag"] = np.where(
    df2["mamografia"] == "sim",
    "Fez",
    "Não fez"
)

def tabela_percentual(df, coluna_flag, origem):
    tab = (
        df[df["origem"] == origem]
        .value_counts(coluna_flag, normalize=True)
        .mul(100)
        .round(2)
        .reindex(["Fez", "Não fez"])
    )
    return tab



In [14]:
tab_2013 = pd.DataFrame({
    "Papanicolau": tabela_percentual(df2, "preventivo_flag", "2013"),
    "Mamografia": tabela_percentual(df2, "mamografia_flag", "2013"),
})

tab_2013.loc["Total"] = 100
tab_2013


,Papanicolau,Mamografia
Fez,86.77,58.27
Não fez,13.23,41.73
Total,100.00,100.00


In [15]:
tab_2019 = pd.DataFrame({
    "Papanicolau": tabela_percentual(df2, "preventivo_flag", "2019"),
    "Mamografia": tabela_percentual(df2, "mamografia_flag", "2019"),
})

tab_2019.loc["Total"] = 100
tab_2019


,Papanicolau,Mamografia
Fez,90.8,64.1
Não fez,9.2,35.9
Total,100.0,100.0
